# 🎯 VISUAL 3 — DCASE SCORE & pAUC@0.1
## *Por que 90% de acurácia não é suficiente para salvar máquinas?*

---

## 🧒 A história do alarme de incêndio

Imagine dois detectores de fumaça:

| | Detector A | Detector B |
|-|-----------|----------|
| Acerta incêndios? | 98% | 95% |
| Alarmes falsos por dia | 10 | 0.5 |
| **Confiança da equipe** | **Baixa** ("é louco esse alarme") | **Alta** |
| **Custo industrial** | 💸 Alto | ✅ Baixo |

**O Detector A tem MAIS acurácia, mas MENOS valor.** Na indústria, cada alarme falso para a linha de produção, manda técnicos investigar, e corrói a confiança no sistema.

É por isso que medimos **pAUC@0.1**: quanto o sistema acerta **quando os alarmes falsos são raros** (< 10% do tempo).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#30363d', 'grid.color': '#30363d'
})
print('Analisando Score DCASE e pAUC@0.1!')

In [ ]:
# ============================================================
# VISUAL — Curva ROC e o que é pAUC
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🎯 O que é pAUC@0.1 e por que o DCASE usa essa métrica?', 
             color='white', fontsize=14, fontweight='bold')

fpr = np.linspace(0, 1, 1000)

# Modelos com curvas ROC distintas
tpr_xgb = 1 - (1 - fpr)**4.5
tpr_gru = 1 - (1 - fpr)**2.8
tpr_random = fpr  # modelo aleatório

ax1 = axes[0]
ax1.set_facecolor('#161b22')

# Região pAUC
fpr_region = fpr[fpr <= 0.1]
ax1.fill_between(fpr_region, tpr_xgb[fpr <= 0.1], alpha=0.4, color='#2ea043', label='pAUC@0.1 do XGBoost')

ax1.plot(fpr, tpr_xgb, color='#2ea043', linewidth=2.5, label='XGBoost+HHT (campeão)')
ax1.plot(fpr, tpr_gru, color='#9e6a03', linewidth=2.5, label='GRU (borderline)')
ax1.plot(fpr, tpr_random, 'k--', linewidth=1.5, alpha=0.5, label='Modelo aleatório (inútil)')

ax1.axvline(0.1, color='#da3633', linewidth=2.5, linestyle='--', label='FPR = 10% (limite DCASE)')

ax1.set_xlabel('FPR — Taxa de Alarmes Falsos\n(← menos alarmes falsos | mais alarmes falsos →)', 
               color='white', fontsize=9)
ax1.set_ylabel('TPR — Taxa de Detecção Real\n(↑ detecta mais anomalias reais ↓)', color='white', fontsize=9)
ax1.set_title('Curva ROC: a "foto" do desempenho de cada detetive\n'
              'A área verde = pAUC@0.1 (região que importa na indústria)', 
              color='white', fontsize=10)
ax1.legend(frameon=False, labelcolor='white', fontsize=9)
ax1.grid(alpha=0.2)

# Anotação explicativa
ax1.text(0.55, 0.25, 
         'Lado esquerdo do gráfico\n= zona industrial\n(poucos alarmes falsos,\nmuita detecção real)',
         ha='center', va='center', color='#8b949e', fontsize=9, style='italic',
         bbox=dict(boxstyle='round', facecolor='#21262d', edgecolor='#30363d'))

# Gráfico 2 — O que muda quando usamos pAUC vs F1
ax2 = axes[1]
ax2.set_facecolor('#161b22')

modelos_pauc = ['XGBoost\n+HHT', 'Tiny-AST', 'GRU', 'GMM', 'Mahal.']
f1_scores = [0.9568, 0.97, 0.88, 0.895, 0.82]
pauc_scores = [0.94, 0.91, 0.87, 0.88, 0.82]
ranking_f1 = [3, 1, 4, 2, 5]
ranking_pauc = [1, 3, 4, 2, 5]

x = np.arange(len(modelos_pauc))
width = 0.35
bars_f1 = ax2.bar(x - width/2, f1_scores, width, label='F1-Score Global', color='#8b949e', alpha=0.85)
bars_pauc = ax2.bar(x + width/2, pauc_scores, width, label='pAUC@0.1 (DCASE)', color='#2ea043', alpha=0.85)

for i, (bar_f1, bar_pauc, mod, r_f1, r_p) in enumerate(zip(bars_f1, bars_pauc, modelos_pauc, ranking_f1, ranking_pauc)):
    f1_v = bar_f1.get_height()
    p_v = bar_pauc.get_height()
    ax2.text(bar_f1.get_x() + bar_f1.get_width()/2, f1_v + 0.003, f'#{r_f1}\n{f1_v:.2f}', 
             ha='center', va='bottom', color='#8b949e', fontsize=8)
    ax2.text(bar_pauc.get_x() + bar_pauc.get_width()/2, p_v + 0.003, f'#{r_p}\n{p_v:.2f}', 
             ha='center', va='bottom', color='#2ea043', fontsize=8)

ax2.axhline(0.80, color='#da3633', linestyle='--', linewidth=2, label='Mínimo DCASE')
ax2.set_xticks(x)
ax2.set_xticklabels(modelos_pauc, color='white')
ax2.set_ylim(0.75, 1.02)
ax2.set_title('⚡ A Virada: Ranking muda quando usamos pAUC!\n'
              'Tiny-AST parecia campeão (#1 em F1) mas é #3 em DCASE', 
              color='white', fontsize=10)
ax2.legend(frameon=False, labelcolor='white', fontsize=9)
ax2.grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUAL — pAUC por fonte de dados (LOSO)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

fontes = ['DCASE\n(sons industriais)', 'MIMII\n(-6dB ruído pesado)', 'MIMII\n(0dB)', 
          'MIMII\n(+6dB limpo)', 'Kaggle\n(rolamentos)', 'Drive Próprio\n(celular)']
pauc_sup = [0.94, 0.87, 0.91, 0.95, 0.93, 0.68]
pauc_unsup = [0.88, 0.82, 0.85, 0.89, 0.87, 0.62]

x = np.arange(len(fontes))
width = 0.35

bars1 = ax.bar(x - width/2, pauc_sup, width, label='Protocolo A (XGBoost)', color='#2ea043', alpha=0.85)
bars2 = ax.bar(x + width/2, pauc_unsup, width, label='Protocolo B (GMM)', color='#1f6feb', alpha=0.85)

ax.axhline(0.80, color='#da3633', linestyle='--', linewidth=2, label='Limite mínimo pAUC (0.80)')

for bars in [bars1, bars2]:
    for bar in bars:
        val = bar.get_height()
        color = '#2ea043' if val >= 0.80 else '#da3633'
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.2f}', 
                ha='center', va='bottom', color=color, fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(fontes, color='white', fontsize=9)
ax.set_ylim(0.55, 1.02)
ax.set_ylabel('pAUC@0.1', color='white', fontsize=11)
ax.set_title('📊 pAUC@0.1 por Fonte de Dados (Protocolo LOSO)\n'
             'Drive Próprio é o mais difícil: microfone de celular em ambiente ruidoso\n'
             '(isso motivou o HHT+UKF e o Mixup)', color='white', fontsize=11)
ax.legend(frameon=False, labelcolor='white', fontsize=10)
ax.grid(alpha=0.2, axis='y')

ax.annotate('⚠️ Mais difícil:\nDomain Shift máximo', xy=(5 - width/2, 0.68), 
            xytext=(4.2, 0.58),
            arrowprops=dict(arrowstyle='->', color='#f0883e', lw=2),
            color='#f0883e', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 🎓 Resumo do Score DCASE 2025

O **Score DCASE** é calculado como:

```
Score DCASE = 0.5 × pAUC_supervisionado + 0.5 × pAUC_não-supervisionado
            = 0.5 × 0.94 + 0.5 × 0.88
            = 0.91
```

| Dataset | Prot. A | Prot. B | Score DCASE |
|---------|---------|---------|-------------|
| DCASE | 0.94 | 0.88 | **0.91** |
| MIMII (médio) | 0.90 | 0.84 | **0.87** |
| Kaggle | 0.93 | 0.87 | **0.90** |
| Drive Próprio | 0.68 | 0.62 | **0.65** |
| **MÉDIA GERAL** | **0.86** | **0.80** | **0.83** |

> **Score DCASE médio de 0.83** — acima do baseline e dentro dos padrões competitivos do DCASE 2025.

---
*Benchmark Visual 3/4 | Projeto AudioAlert | Emanoel Spanhol | UniSENAI 2025-2026*